In [5]:
import re
import spacy
from spacy.matcher import PhraseMatcher
from huggingface_hub import snapshot_download
from rapidfuzz import process


# =========================
# 1. LOAD MODELS
# =========================
def load_skill_model():
    model_path = snapshot_download("amjad-awad/skill-extractor", repo_type="model")
    return spacy.load(model_path)

def load_name_model():
    return spacy.load("en_name_model")

def load_exp_model():
    return spacy.load("en_experience_model")


# =========================
# 2. LOAD SKILL DICTIONARY
# =========================
def load_skill_dictionary(file_path, nlp):
    with open(file_path, "r") as f:
        skills = [line.strip() for line in f if line.strip()]
    
    matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
    patterns = [nlp.make_doc(skill) for skill in skills]
    matcher.add("SKILL_DB", patterns)

    normalized_db = [normalize(s) for s in skills]

    return matcher, normalized_db


# =========================
# 3. NORMALIZATION
# =========================
def normalize(skill):
    skill = skill.lower().strip()
    skill = re.sub(r"[^\w\s]", " ", skill)
    skill = re.sub(r"\s+", " ", skill)
    return skill


# =========================
# 4. CANONICAL MAPPING
# =========================
CANONICAL_MAP = {
    "js": "javascript",
    "react js": "react",
    "reactjs": "react",
    "node": "nodejs",
    "node js": "nodejs",
    "ml": "machine learning",
    "dl": "deep learning",
    "nlp": "natural language processing",
    "front end": "frontend",
    "sklearn": "scikit learn",
    "c plus plus": "c++"
    
}

GENERALIZATION_MAP = {
    "database management systems": "database",
    "database management": "database",
    "dbms": "database",
    "object oriented programming": "oop",
    "oops": "oop",
    "data structures and algorithms": "dsa",
    "data structures": "dsa",
    "algorithms": "dsa",
    "artificial intelligence": "ai",
    "machine learning": "ml",
    "deep learning": "dl",
    "natural language processing": "nlp"
}

def canonicalize(skill):
    norm = normalize(skill)
    norm = CANONICAL_MAP.get(norm, norm)
    norm = GENERALIZATION_MAP.get(norm, norm)
    return norm


# =========================
# 5. FUZZY MATCHING
# =========================
def fuzzy_map(skill, skill_db):
    match, score, _ = process.extractOne(skill, skill_db)
    return match if score > 88 else skill


# =========================
# 6. EXTRACT SKILLS
# =========================
def is_better_skill(new, existing):
    if len(new.split()) != len(existing.split()):
        return len(new.split()) > len(existing.split())
    return len(new) > len(existing)


def extract_skills(text, nlp, matcher, skill_db):
    doc = nlp(text)

    model_skills = [ent.text for ent in doc.ents if "SKILL" in ent.label_]
    matches = matcher(doc)
    dict_skills = [doc[start:end].text for _, start, end in matches]

    final_skills = {}

    for skill in dict_skills:
        canon = canonicalize(skill)
        if canon not in final_skills or is_better_skill(skill, final_skills[canon]):
            final_skills[canon] = skill

    for skill in model_skills:
        norm = normalize(skill)
        norm = fuzzy_map(norm, skill_db)
        canon = canonicalize(norm)

        if canon not in final_skills or is_better_skill(skill, final_skills[canon]):
            final_skills[canon] = skill

    return list(final_skills.values())


# =========================
# 7. EXTRACT EMAIL
# =========================
def extract_email(text):
    match = re.search(r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+", text)
    return match.group(0) if match else None


# =========================
# 8. EXTRACT NAME
# =========================
def extract_name(text, nlp_name, nlp_fallback):
    # ---------- 1. Try custom model ----------
    doc = nlp_name(text)

    for ent in doc.ents:
        if ent.label_ in ["PERSON", "NAME"]:
            name = ent.text.strip()

            # basic validation
            if (
                len(name.split()) >= 2 and
                not any(char.isdigit() for char in name)
            ):
                return name

    # ---------- 2. Fallback to spaCy model ----------
    doc_fb = nlp_fallback(text)

    for ent in doc_fb.ents:
        if ent.label_ == "PERSON":
            name = ent.text.strip()

            # stronger filtering to avoid "Java"
            words = name.split()

            if (
                2 <= len(words) <= 4 and
                all(w.isalpha() for w in words) and
                not any(char.isdigit() for char in name)
            ):
                return name

    return None


def load_fallback_name_model():
    return spacy.load("en_core_web_sm")

# =========================
# 9. EXTRACT EXPERIENCE (NEW)
# =========================
def extract_experience(text, nlp_exp):
    doc = nlp_exp(text)

    experiences = []

    for ent in doc.ents:
        if ent.label_.lower() == "experience":
            experiences.append(ent.text.strip())

    return experiences


# =========================
# 10. COMBINED EXTRACTION
# =========================
nlp_fallback = load_fallback_name_model()
def extract_all(text, nlp_skill, matcher, skill_db, nlp_name, nlp_exp, nlp_fallback):

    name = extract_name(text, nlp_name, nlp_fallback)
    email = extract_email(text)

    raw_skills = extract_skills(text, nlp_skill, matcher, skill_db)
    skills = list(set([canonicalize(s) for s in raw_skills]))

    experience = extract_experience(text, nlp_exp)

    return {
        "name": name,
        "email": email,
        "skills": skills,
        "experience": experience
    }


# =========================
# 11. MAIN
# =========================
if __name__ == "__main__":

    text = """Saroj Kumar Padhi
Full Stack Developer
Mobile No. - +91 9078055208
2001sarojpadhi25@gmail.com

https://www.linkedin.com/in/saroj-padhi-50024a189/

Enthusiastic and Detail-oriented Full Stack Developer with 2 years of experience in building scalable, responsive web
applications using React, Node.js, Express.js, and MongoDB. Eager to contribute to a tech-driven organization by enhancing
user experience, optimizing performance, and growing within a dynamic team environment.
EXPERIENCE :1. Mentroz Private Ltd. - Software Developer (Sept 2023 - Currently)
➢ Developed and maintained full-stack web applications using the MERN stack (MongoDB, Express.js, React.js,
Node.js) for Mentroz's user-facing platform and company website.
➢ Built reusable, scalable front-end components with React.js, ensuring seamless integration with backend APIs and
real-time data flows.
➢ Collaborated with cross-functional teams to implement features like 1-on-1 mentorship scheduling, career
planning tools, and scholarship listings.
➢ Wrote modular backend APIs using Node.js and Express, handling user authentication, mentor matching, and
secure data transactions.
➢ Worked with Figma and modern design systems to translate wireframes into interactive front-end components.
➢ Managed code using Git, participated in code reviews, and followed Agile practices including sprint planning and
daily stand-ups.
2. Tetra Trion Technology Private Ltd. - Android App Internship (January - May 2023 )
➢ Collaborated on the development of cross-platform mobile applications using React Native, targeting Android
devices with scalable, performant code.
➢ Integrated RESTful APIs and third-party libraries to enhance app functionality and deliver real-time data.
➢ Participated in the full app lifecycle including UI/UX design.
➢ Used tools like Expo, Android Studio, VS Code, and Git for efficient development and version control.
PROJECTS :1. Mentroz : Mentroz Career Counseling Platform offers a structured environment for individuals to receive one-to-one
mentorship and personalized career planning support. Key features include mentor browsing, real-time session booking,
scholarship, profile management, and tailored guidance to help users set and achieve their career goals.
2. RoutEasy : Commercial bus ticket booking Application. Which provides Travel experience for users, the Application offer a
user-friendly user interface for Browsing schedules, Selecting seats and making secure payments and currently I’m still
working on it in Frontend part.
3. Websites : I have completed many small projects like Registration form, Sign-Up page, Commercial shopping Pages and
also many cards.
SKILLS :• Programming Languages: C language
• Mobile development : Android Studio
• Tools & Platforms: Git/GitHub, Postman, VScode,
Figma, Chatgpt
• Productivity Tools: MS Word, MS Excel, MS Power
Point
• Strong Proficiency in Java Script includes DOM
Manipulation and Object model

Front-end: HTML5, CSS3, Jave Script, React.js, ReactNative, Bootstrap Components, Material UI
Backend: Node.js, Express.js
Database: MySQL, MongoDB,
I have Strong foundation in MERN Stack
Ability to understand Logics, Functionality and Project
Flow

EDUCATION :Master of Computer Applications (MCA)
2022-24
NIIS Institute of Business Administration, Odisha
Bachelor of science in Mathematics hons.
2019-22
Sri Jaydev college of Education and Technology,Odisha
12th Science
2017-19
Maharishi College of Natural Law, Bhubaneswar, Odisha

CERTIFICATES :• UI full Stack with React
• Core Java
• Web Designing Course Word press Application
• Android App Development
• Tally with GST Advance

SOFT SKILLS :• Teamwork Skills
• Good leadership
• Calm & tolerant
• Positive Attribute
• Problem solving

INTERESTS
• Travelling
• Listening music
• Bike ride
"""

    # Load models
    nlp_skill = load_skill_model()
    nlp_name = load_name_model()
    nlp_exp = load_exp_model()

    # Load dictionary
    matcher, skill_db = load_skill_dictionary("unique_skills.txt", nlp_skill)

    # Extract everything
    result = extract_all(text, nlp_skill, matcher, skill_db, nlp_name, nlp_exp, nlp_fallback)

    print("\n===== EXTRACTED INFO =====\n")
    print("Name:", result["name"])
    print("Email:", result["email"])
    print("Skills:", result["skills"])
    print("Experience:", result["experience"])

Fetching 15 files: 100%|████████████████████| 15/15 [00:00<00:00, 132451.71it/s]



===== EXTRACTED INFO =====

Name: Saroj Kumar Padhi
Email: 2001sarojpadhi25@gmail.com
Skills: ['java script', 'git github', 'ms excel', 'nodejs', 'restful apis', 'react', 'full stack', 'mongodb', 'html5', 'react native', 'productivity', 'ui', 'agile', 'css3', 'authentication', 'java', 'code reviews', 'code', 'problem solving', 'leadership', 'ms power', 'database', 'user experience', 'github', 'mysql', 'teamwork', 'excel', 'android studio', 'mathematics', 'bootstrap', 'backend', 'express js', 'frontend', 'figma', 'dom', 'git', 'user interface', 'version control', 'web applications', 'programming', 'postman', 'ui ux', 'mern', 'ms word', 'bhubaneswar', 'apis']
Experience: ['Full Stack Developer with 2 years of experience', 'Mentroz Private Ltd. - Software Developer (Sept 2023 - Currently)', 'Android App Internship (January - May 2023 )']
